In [1]:
 # ===== 1. 환경 / 세팅 =====
import os, json, gc, sys, math, random, subprocess
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.feature_selection import VarianceThreshold
from dataclasses import dataclass, asdict
from typing import List, Tuple, Dict, Optional

# Colab Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Colab 외 환경이면 무시:", e)

DATA_DIR = "/content/drive/MyDrive/data/instacart"
INPUT_CSV = os.path.join(DATA_DIR, "master_dataset_with_roles_final.csv")
OUT_DIR   = DATA_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# 속도/재현 세팅
RANDOM_SEED = 2025
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# GPU 사용 가능 체크 → xgboost 3.0.4에서 'gpu_hist' 권장
def detect_tree_method() -> str:
    try:
        _ = subprocess.check_output(["nvidia-smi"])
        return "gpu_hist"
    except Exception:
        return "hist"

TREE_METHOD = detect_tree_method()
print("TREE_METHOD:", TREE_METHOD)

# xgboost 3.0.4 확인
import xgboost as xgb
print("xgboost version:", xgb.__version__)
assert xgb.__version__.startswith("3.0.4"), "환경의 xgboost 버전이 3.0.4가 아닙니다. (변경 불가라면 이 경고를 무시하지 마세요)"

Mounted at /content/drive
TREE_METHOD: gpu_hist
xgboost version: 3.0.4


In [2]:
# ===== 2. 유틸 =====

def first_existing(cands: List[str], cols: List[str]) -> Optional[str]:
    for c in cands:
        if c in cols:
            return c
    return None

def is_binary_series(s: pd.Series) -> bool:
    vals = pd.unique(s.dropna())
    return set(vals.tolist()).issubset({0,1}) or set(vals.tolist()).issubset({0.0,1.0})

# Instacart F1: 주문 단위 평균, 빈 예측은 'None' 처리
def f1_single(true_set: set, pred_set: set) -> float:
    if len(true_set) == 0 and len(pred_set) == 0:
        return 1.0
    if len(pred_set) == 0:
        pred_set = {"None"}
    if len(true_set) == 0:
        true_set = {"None"}
    tp = len(true_set & pred_set)
    if tp == 0:
        return 0.0
    prec = tp / len(pred_set)
    rec  = tp / len(true_set)
    return 0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec)

def order_level_f1(df: pd.DataFrame, order_col: str, product_col: str,
                   target_col: str, proba_col: str, thr: float) -> float:
    f1s = []
    for oid, g in df.groupby(order_col):
        true_set = set(g.loc[g[target_col]==1, product_col].tolist())
        pred_set = set(g.loc[g[proba_col]>=thr, product_col].tolist())
        f1s.append(f1_single(true_set, pred_set))
    return float(np.mean(f1s)) if len(f1s) else 0.0

def search_best_threshold(df: pd.DataFrame, order_col: str, product_col: str,
                          target_col: str, proba_col: str,
                          lo=0.05, hi=0.95, step=0.01) -> Tuple[float, float]:
    best_thr, best_f1 = 0.5, -1.0
    thr = lo
    while thr <= hi + 1e-12:
        f1 = order_level_f1(df, order_col, product_col, target_col, proba_col, thr)
        if f1 > best_f1:
            best_thr, best_f1 = thr, f1
        thr += step
    return best_thr, best_f1


In [4]:
# ===== 3. 데이터 로딩 & 역할 분리 (robust; eval_set 우선) =====
df = pd.read_csv(INPUT_CSV, low_memory=False)
print("원본 shape:", df.shape)
print("컬럼 예시:", df.columns.tolist()[:40])

cols = df.columns.tolist()

# 주요 컬럼 자동 감지
target_col = first_existing(['reordered','label','target','y','is_reordered'], cols)
member_col = first_existing(['member_id','user_id','uid','user'], cols)
product_col= first_existing(['product_id','pid','product'], cols)
order_col  = first_existing(['order_id','oid','order'], cols)
eval_col   = 'eval_set' if 'eval_set' in cols else None

if target_col is None:
    raise ValueError("타깃 컬럼(reordered/label/target/y/is_reordered 중 1개)이 필요합니다.")
if product_col is None:
    raise ValueError("product_id(또는 유사) 컬럼이 필요합니다.")
if (member_col is None) and (order_col is None):
    raise ValueError("member_id 또는 order_id 중 하나는 반드시 필요합니다.")

# 역할 컬럼 후보
role_train_col = 'role_train' if 'role_train' in cols else None
role_test_col  = 'role_test'  if 'role_test'  in cols else None
role_col = None if (role_train_col or role_test_col) else first_existing(['role','set','split'], cols)

# Instacart 원본 규칙 반영:
# - 7번 주문(캐글의 eval_set=='test')은 평가 서버용이므로 제거
if eval_col is not None:
    ev = df[eval_col].astype(str).str.lower()
    if 'test' in set(ev.unique()):
        before = len(df)
        df = df.loc[~ev.eq('test')].copy()
        print(f"eval_set=='test' 행 {before - len(df):,}개 제거(7번 주문 삭제).")
    # 분포 출력
    ev = df[eval_col].astype(str).str.lower()
    print("eval_set 분포:", ev.value_counts().to_dict())

# 역할 마스크 생성 함수 (role_*이 망가졌을 때도 동작)
def build_role_mask(_df: pd.DataFrame):
    # 1) role_train/role_test가 있으면 우선 사용 (숫자/불리언/문자 모두 허용)
    if role_train_col and role_test_col:
        a = _df[role_train_col]
        b = _df[role_test_col]

        def to_mask(s, positive_tokens):
            if pd.api.types.is_bool_dtype(s):
                return s.astype(bool)
            if pd.api.types.is_numeric_dtype(s):
                return pd.to_numeric(s, errors='coerce').fillna(0) > 0.5
            # 문자열 처리
            ls = s.astype(str).str.lower()
            return ls.isin(positive_tokens)

        # 문자열로 'prior'/'train'이 들어온 경우도 커버
        tr = to_mask(a, {'1','true','t','y','yes','prior','train'})
        te = to_mask(b, {'1','true','t','y','yes','train'})

        # 둘 다 비정상(전부 False 등)이면 eval_set로 대체
        if (tr.sum()==0 and te.sum()==0) or tr.equals(te):
            print("role_* 해석이 애매하여 eval_set로 대체합니다.")
        else:
            return tr, te

    # 2) eval_set이 있으면: prior=학습(1~5), train=성능평가(6)
    if eval_col is not None:
        lc = _df[eval_col].astype(str).str.lower()
        tr = lc.eq('prior')   # 학습 시 사용
        te = lc.eq('train')   # 성능 평가 시 사용(6번)
        return tr, te

    # 3) role 문자열 컬럼이 있으면 보수적으로 분리
    if role_col:
        lc = _df[role_col].astype(str).str.lower()
        tr = lc.isin(['train','training','role_train','tr','prior'])
        te = lc.isin(['test','role_test','validation','val','eval','train'])
        return tr, te

    # 4) 마지막 폴백: 전부 학습으로 처리(비권장)
    print("⚠️ 역할 컬럼을 못 찾았어요. 전체를 train으로 처리합니다.")
    n = len(_df)
    return pd.Series([True]*n, index=_df.index), pd.Series([False]*n, index=_df.index)

train_mask, test_mask = build_role_mask(df)
df_train = df.loc[train_mask].copy()
df_test  = df.loc[test_mask].copy()

print("train rows:", df_train.shape, "| test rows:", df_test.shape)

# 타깃 이진 보정
if not is_binary_series(df_train[target_col]):
    df_train[target_col] = (df_train[target_col] > 0).astype(np.int8)
    if target_col in df_test.columns:
        df_test[target_col]  = (df_test[target_col]  > 0).astype(np.int8)



원본 shape: (33819106, 17)
컬럼 예시: ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department', 'role_train', 'role_test']
eval_set=='test' 행 11,792,498개 제거(7번 주문 삭제).
eval_set 분포: {'prior': 20641991, 'train': 1384617}
train rows: (20641991, 17) | test rows: (1384617, 17)


In [5]:
# ===== 4. 피처 구성 & 전처리 =====

# 제외할 컬럼
exclude = {c for c in [target_col, member_col, product_col, order_col,
                       role_col, role_train_col, role_test_col] if c}

num_cols = [c for c in df.columns if (c not in exclude) and pd.api.types.is_numeric_dtype(df[c])]
print("원시 수치 피처 개수:", len(num_cols))

# 결측치 & 다운캐스트
for c in num_cols:
    df_train[c] = df_train[c].astype('float32')
    df_test[c]  = df_test[c].astype('float32')
df_train[num_cols] = df_train[num_cols].fillna(-1.0)
df_test[num_cols]  = df_test[num_cols].fillna(-1.0)

# (선택) 아주 작은 분산 피처 제거 → 속도 향상
REMOVE_LOW_VAR = True
if REMOVE_LOW_VAR:
    vt = VarianceThreshold(threshold=1e-10)
    vt.fit(df_train[num_cols].values)
    keep_idx = np.where(vt.get_support())[0].tolist()
    num_cols = [num_cols[i] for i in keep_idx]
    print("저분산 제거 후 수치 피처:", len(num_cols))

# 모델 A/B용 피처셋
base_keywords = ['count','ratio','rate','reorder','recency','since',
                 'dow','hour','days','streak','freq','interval',
                 'avg','mean','sum','var','std','last','n_',
                 'cum','rank','dept','aisle','cart','position']
features_A = sorted({c for c in num_cols if any(k in c.lower() for k in base_keywords)})

# 너무 적으면 num_cols에서 보강
if len(features_A) < max(20, len(num_cols)//6):
    rng = np.random.default_rng(RANDOM_SEED)
    add = rng.choice(num_cols, size=min(80, len(num_cols)), replace=False).tolist()
    features_A = sorted(set(features_A + add))

features_B = sorted(num_cols)

print(f"Model A features: {len(features_A)} | Model B features: {len(features_B)}")


원시 수치 피처 개수: 7
저분산 제거 후 수치 피처: 7
Model A features: 7 | Model B features: 7


In [9]:
# ===== 5. 학습 (3-fold OOF) & 앙상블 — Booster API (xgboost 3.0.4 호환) =====
from dataclasses import dataclass
from typing import List, Optional, Dict
import numpy as np
import pandas as pd
import os, gc
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss
import xgboost as xgb

# --- sklearn 버전 호환: eps 없이 log_loss를 쓰되, 직접 clip ---
def safe_log_loss(y_true, y_pred):
    p = np.clip(np.asarray(y_pred, dtype=np.float64), 1e-15, 1 - 1e-15)
    return float(log_loss(y_true, p))

@dataclass
class XGBCfg:
    n_estimators: int
    max_depth: int
    learning_rate: float
    subsample: float
    colsample_bytree: float
    min_child_weight: float
    reg_lambda: float
    gamma: float
    tree_method: str        # 'gpu_hist' or 'hist' 등 (외부에서 넘어옴)
    random_state: int

# 속도 고려한 기본값 (필요시 조정)
NFOLDS = 3
cfg_A = XGBCfg(
    n_estimators=500, max_depth=7, learning_rate=0.05,
    subsample=0.90, colsample_bytree=0.90, min_child_weight=1.0,
    reg_lambda=1.0, gamma=0.0, tree_method=TREE_METHOD, random_state=42
)
cfg_B = XGBCfg(
    n_estimators=650, max_depth=6, learning_rate=0.035,
    subsample=0.85, colsample_bytree=0.85, min_child_weight=5.0,
    reg_lambda=1.5, gamma=0.1, tree_method=TREE_METHOD, random_state=2025
)

def _bst_predict_proba(bst: xgb.Booster, dmat: xgb.DMatrix) -> np.ndarray:
    """best_iteration을 반영해 확률 반환 (3.x: iteration_range; 필요시 ntree_limit fallback)."""
    bi = getattr(bst, "best_iteration", None)
    if bi is not None:
        try:
            preds = bst.predict(dmat, iteration_range=(0, int(bi) + 1))
            return preds.astype(np.float32)
        except TypeError:
            pass
        # (구버전 fallback)
        try:
            preds = bst.predict(dmat, ntree_limit=getattr(bst, "best_ntree_limit", int(bi) + 1))
            return preds.astype(np.float32)
        except Exception:
            pass
    return bst.predict(dmat).astype(np.float32)

def fit_xgb_cv_booster(
    df_tr: pd.DataFrame,
    features: List[str],
    target: str,
    groups: Optional[pd.Series],
    cfg: XGBCfg,
    model_prefix: str,
    n_splits: int = NFOLDS,
    early_stopping_rounds: int = 50
) -> Dict:
    X = df_tr[features].values
    y = df_tr[target].values.astype(np.float32)

    if groups is None:
        groups = np.arange(len(y)) % n_splits
    splitter = GroupKFold(n_splits=n_splits)

    oof = np.zeros(len(df_tr), dtype=np.float32)
    fold_models: List[xgb.Booster] = []
    fold_metrics = []

    # device 매핑: gpu_hist → device='cuda', 아니면 cpu
    device = "cuda" if cfg.tree_method == "gpu_hist" else "cpu"

    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "max_depth": cfg.max_depth,
        "eta": cfg.learning_rate,          # learning_rate alias
        "subsample": cfg.subsample,
        "colsample_bytree": cfg.colsample_bytree,
        "min_child_weight": cfg.min_child_weight,
        "lambda": cfg.reg_lambda,
        "gamma": cfg.gamma,
        "tree_method": "hist",             # 3.x 권장: hist + device
        "device": device,
        "verbosity": 1,
        "seed": cfg.random_state,
    }

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y, groups=groups), 1):
        dtr = xgb.DMatrix(X[tr_idx], label=y[tr_idx], feature_names=features)
        dva = xgb.DMatrix(X[va_idx], label=y[va_idx], feature_names=features)

        bst = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=cfg.n_estimators,
            evals=[(dva, "valid")],      # early stopping은 valid 기준
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )

        best_iter = getattr(bst, "best_iteration", None)
        proba_va = _bst_predict_proba(bst, dva)
        oof[va_idx] = proba_va

        auc = roc_auc_score(y[va_idx], proba_va) if len(np.unique(y[va_idx])) > 1 else np.nan
        ll  = safe_log_loss(y[va_idx], proba_va)
        print(f"[{model_prefix}] Fold {fold} | AUC={auc:.5f} | Logloss={ll:.5f} | BestIter={best_iter}")

        # 모델 저장
        bst.save_model(os.path.join(OUT_DIR, f"{model_prefix}_fold{fold}.json"))
        fold_models.append(bst)
        fold_metrics.append({
            "fold": fold,
            "auc": float(auc),
            "logloss": float(ll),
            "best_iteration": int(best_iter if best_iter is not None else 0)
        })

        del dtr, dva
        gc.collect()

    return {"oof": oof, "models": fold_models, "metrics": fold_metrics, "features": features}

# === 학습 실행 ===
groups = df_train[member_col] if member_col else None
res_A = fit_xgb_cv_booster(df_train, features_A, target_col, groups, cfg_A, "xgb_A")
res_B = fit_xgb_cv_booster(df_train, features_B, target_col, groups, cfg_B, "xgb_B")

df_train["proba_A"]    = res_A["oof"]
df_train["proba_B"]    = res_B["oof"]
df_train["proba_ens"]  = (df_train["proba_A"] + df_train["proba_B"]) / 2.0

[xgb_A] Fold 1 | AUC=0.79667 | Logloss=0.51865 | BestIter=499
[xgb_A] Fold 2 | AUC=0.79637 | Logloss=0.51922 | BestIter=499
[xgb_A] Fold 3 | AUC=0.79656 | Logloss=0.51895 | BestIter=499
[xgb_B] Fold 1 | AUC=0.79634 | Logloss=0.51903 | BestIter=649
[xgb_B] Fold 2 | AUC=0.79607 | Logloss=0.51956 | BestIter=649
[xgb_B] Fold 3 | AUC=0.79628 | Logloss=0.51927 | BestIter=649


In [11]:
# 긴 연산 전에 OOF 저장 (안전장치)
oof_path = os.path.join(OUT_DIR, "oof_predictions_instacart.csv")
cols_to_save = [c for c in [order_col, member_col, product_col] if c] + [target_col, "proba_A","proba_B","proba_ens"]
(df_train[cols_to_save]).to_csv(oof_path, index=False)
print("OOF 임시 저장:", oof_path, "| rows:", len(df_train))

OOF 임시 저장: /content/drive/MyDrive/data/instacart/oof_predictions_instacart.csv | rows: 20641991


In [12]:
import numpy as np
import pandas as pd
import time

# 빠른 임계치 탐색: 주문 샘플 + 분위수 후보 + 진행률 출력
def search_best_threshold_fast(
    df: pd.DataFrame,
    order_col: str,
    product_col: str,
    target_col: str,
    proba_col: str,
    quantile_lo: float = 0.05,
    quantile_hi: float = 0.95,
    n_candidates: int = 31,
    max_orders: int = 50_000,     # 샘플링할 최대 주문 수 (속도 핵심 파라미터)
    seed: int = 2025,
    verbose: bool = True
):
    t0 = time.time()
    # 1) 주문 샘플링
    orders = df[order_col].drop_duplicates()
    if (max_orders is not None) and (len(orders) > max_orders):
        sampled_orders = orders.sample(max_orders, random_state=seed)
        sub = df[df[order_col].isin(sampled_orders)][[order_col, product_col, target_col, proba_col]].copy()
        if verbose:
            print(f"[thr-search] sample orders: {len(sampled_orders):,}  rows: {len(sub):,}")
    else:
        sub = df[[order_col, product_col, target_col, proba_col]].copy()
        if verbose:
            print(f"[thr-search] full orders: {sub[order_col].nunique():,}  rows: {len(sub):,}")

    # 2) 후보 임계치: 분위수 기반
    qs = np.linspace(quantile_lo, quantile_hi, n_candidates)
    thr_list = np.unique(sub[proba_col].quantile(qs).values)
    if verbose:
        print(f"[thr-search] candidates: {len(thr_list)}  (quantiles {quantile_lo:.2f}~{quantile_hi:.2f})")

    # 3) 주문별 실제 양성 개수(고정)
    true_cnt = sub.groupby(order_col)[target_col].sum().astype(np.int32)

    best_thr, best_f1 = 0.5, -1.0
    for i, t in enumerate(thr_list, 1):
        # 예측/정답 교차 카운트 (임계치마다 1회 groupby)
        mask = (sub[proba_col] >= t)
        pred_cnt = sub.loc[mask].groupby(order_col, observed=True)[proba_col].size()
        tp_cnt   = sub.loc[mask & (sub[target_col] == 1)].groupby(order_col, observed=True)[target_col].size()

        # 누락 주문 0 채우기
        agg = pd.DataFrame({
            'true': true_cnt,
            'pred': pred_cnt.reindex(true_cnt.index, fill_value=0).astype(np.int32),
            'tp'  : tp_cnt.reindex(true_cnt.index,  fill_value=0).astype(np.int32),
        })

        # 주문 단위 F1 (None 규칙 포함)
        true = agg['true'].values
        pred = agg['pred'].values
        tp   = agg['tp'].values

        none_case = (pred == 0) & (true == 0)

        with np.errstate(divide='ignore', invalid='ignore'):
            precision = np.divide(tp, pred, out=np.zeros_like(tp, dtype=float), where=pred>0)
            recall    = np.divide(tp, true, out=np.zeros_like(tp, dtype=float), where=true>0)
            denom     = precision + recall
            f1_arr    = np.divide(2*precision*recall, denom, out=np.zeros_like(denom), where=denom>0)
        f1_arr[none_case] = 1.0

        f1 = float(f1_arr.mean())
        if verbose and (i % max(1, len(thr_list)//5) == 0 or i == len(thr_list)):
            elapsed = time.time() - t0
            print(f"[thr-search] {i}/{len(thr_list)}  thr={t:.4f}  F1={f1:.5f}  elapsed={elapsed:.1f}s")

        if f1 > best_f1:
            best_f1, best_thr = f1, float(t)

    if verbose:
        print(f"[thr-search] best_thr={best_thr:.4f}  best_f1={best_f1:.5f}  total={time.time()-t0:.1f}s")
    return best_thr, best_f1

# 큰 배열에서 AUC/Logloss가 너무 무겁다면 표본으로 계산
def fast_auc_logloss(df: pd.DataFrame, y_col: str, p_col: str, max_rows: int = 2_000_000, seed: int = 2025):
    y = df[y_col].to_numpy()
    p = np.clip(df[p_col].to_numpy(dtype=np.float64), 1e-15, 1-1e-15)
    if len(y) > max_rows:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(y), size=max_rows, replace=False)
        y = y[idx]; p = p[idx]
        print(f"[metrics] sampled {len(y):,} rows for AUC/Logloss")
    auc = roc_auc_score(y, p) if len(np.unique(y))>1 else np.nan
    from sklearn.metrics import log_loss as _ll
    ll  = float(_ll(y, p))
    return auc, ll

In [13]:
# ===== 6. OOF 성능평가 & 저장 — fast threshold search =====
report = {}
order_key = order_col if order_col else member_col

EVAL_ONLY_ENSEMBLE = True           # 속도 위해 앙상블만 임계치 탐색
N_CANDIDATES = 31                   # 분위수 후보 개수 (31~41 추천)
MAX_ORDERS_FOR_THR = 50_000         # 임계치 탐색에 사용할 최대 주문 수
MAX_ROWS_FOR_AUC_LL = 2_000_000     # AUC/Logloss 표본 크기

def eval_block_fast(df_eval, proba_col, label=""):
    thr, f1 = search_best_threshold_fast(
        df_eval, order_key, product_col, target_col, proba_col,
        n_candidates=N_CANDIDATES, max_orders=MAX_ORDERS_FOR_THR, verbose=True
    )
    auc, ll = fast_auc_logloss(df_eval, target_col, proba_col, max_rows=MAX_ROWS_FOR_AUC_LL)
    print(f"{label} | BestThr={thr:.4f} | F1={f1:.5f} | AUC≈{auc:.5f} | Logloss≈{ll:.5f}")
    return {'best_thr': float(thr), 'f1': float(f1), 'auc': float(auc), 'logloss': float(ll)}

# (선택) A/B도 간단히 표본 AUC/LL만 출력
def eval_block_auc_only(df_eval, proba_col, label=""):
    auc, ll = fast_auc_logloss(df_eval, target_col, proba_col, max_rows=MAX_ROWS_FOR_AUC_LL)
    print(f"{label} | AUC≈{auc:.5f} | Logloss≈{ll:.5f}")
    return {'best_thr': None, 'f1': None, 'auc': float(auc), 'logloss': float(ll)}

if EVAL_ONLY_ENSEMBLE:
    report['model_A']  = eval_block_auc_only(df_train, "proba_A", "[OOF] Model A")
    report['model_B']  = eval_block_auc_only(df_train, "proba_B", "[OOF] Model B")
    report['ensemble'] = eval_block_fast(df_train, "proba_ens", "[OOF] Ensemble")
else:
    # 느리지만 A/B도 임계치 탐색
    report['model_A']  = eval_block_fast(df_train, "proba_A", "[OOF] Model A")
    report['model_B']  = eval_block_fast(df_train, "proba_B", "[OOF] Model B")
    report['ensemble'] = eval_block_fast(df_train, "proba_ens", "[OOF] Ensemble")

best_thr = report['ensemble']['best_thr']

# 산출물 저장
oof_path = os.path.join(OUT_DIR, "oof_predictions_instacart.csv")
cols_to_save = [c for c in [order_col, member_col, product_col] if c] + [target_col, "proba_A","proba_B","proba_ens"]
df_train[cols_to_save].to_csv(oof_path, index=False)

meta_path = os.path.join(OUT_DIR, "ensemble_meta.json")
with open(meta_path, "w") as f:
    json.dump({
        'tree_method': 'hist',
        'device': 'cuda' if cfg_A.tree_method=='gpu_hist' else 'cpu',
        'cfg_A': asdict(cfg_A),
        'cfg_B': asdict(cfg_B),
        'features_A': res_A['features'],
        'features_B': res_B['features'],
        'report_oof': report,
        'best_threshold': best_thr,
        'columns': {
            'target_col': target_col,
            'order_col': order_col,
            'member_col': member_col,
            'product_col': product_col,
            'role_col': role_col,
            'role_train_col': role_train_col,
            'role_test_col': role_test_col
        }
    }, f, indent=2, ensure_ascii=False)

print("OOF/메타 저장 완료:", oof_path, meta_path)

[metrics] sampled 2,000,000 rows for AUC/Logloss
[OOF] Model A | AUC≈0.79592 | Logloss≈0.51965
[metrics] sampled 2,000,000 rows for AUC/Logloss
[OOF] Model B | AUC≈0.79558 | Logloss≈0.52003
[thr-search] sample orders: 50,000  rows: 501,646
[thr-search] candidates: 31  (quantiles 0.05~0.95)
[thr-search] 6/31  thr=0.3656  F1=0.74914  elapsed=1.1s
[thr-search] 12/31  thr=0.5607  F1=0.71066  elapsed=1.3s
[thr-search] 18/31  thr=0.6915  F1=0.61880  elapsed=1.4s
[thr-search] 24/31  thr=0.7900  F1=0.47681  elapsed=1.6s
[thr-search] 30/31  thr=0.8823  F1=0.26755  elapsed=1.7s
[thr-search] 31/31  thr=0.9021  F1=0.21923  elapsed=1.7s
[thr-search] best_thr=0.3209  best_f1=0.75068  total=1.7s
[metrics] sampled 2,000,000 rows for AUC/Logloss
[OOF] Ensemble | BestThr=0.3209 | F1=0.75068 | AUC≈0.79583 | Logloss≈0.51975
OOF/메타 저장 완료: /content/drive/MyDrive/data/instacart/oof_predictions_instacart.csv /content/drive/MyDrive/data/instacart/ensemble_meta.json


In [14]:
# ===== 7. role_test 예측 & 평가 / 제출 — Booster API 버전 =====

def predict_with_models(df_in: pd.DataFrame, features: List[str], models: List[xgb.Booster], out_col: str):
    X = df_in[features].values.astype(np.float32)
    dmat = xgb.DMatrix(X, feature_names=features)
    preds = np.zeros(len(df_in), dtype=np.float32)
    for bst in models:
        preds += _bst_predict_proba(bst, dmat)
    preds /= max(1, len(models))
    df_in[out_col] = preds
    return preds

if len(df_test) == 0:
    print("⚠️ role_test 데이터가 없습니다.")
else:
    predict_with_models(df_test, res_A['features'], res_A['models'], "proba_A")
    predict_with_models(df_test, res_B['features'], res_B['models'], "proba_B")
    df_test["proba_ens"] = (df_test["proba_A"] + df_test["proba_B"]) / 2.0

    has_label_test = (target_col in df_test.columns) and is_binary_series(df_test[target_col])
    if has_label_test:
        f1  = order_level_f1(df_test, order_key, product_col, target_col, "proba_ens", best_thr)
        auc = roc_auc_score(df_test[target_col], df_test["proba_ens"]) if len(np.unique(df_test[target_col]))>1 else np.nan
        ll  = safe_log_loss(df_test[target_col], df_test["proba_ens"])
        print(f"[TEST] Ensemble | Thr={best_thr:.2f} | F1={f1:.5f} | AUC={auc:.5f} | Logloss={ll:.5f}")
    else:
        print("role_test 라벨이 없어 F1/AUC 계산 생략. 제출 포맷만 생성합니다.")

    # 주문별 예측 products 문자열 생성 (Kaggle 포맷: 비어있으면 'None')
    def to_pred_string(g: pd.DataFrame, thr: float) -> str:
        items = g.loc[g["proba_ens"] >= thr, product_col].astype(str).tolist()
        return "None" if len(items) == 0 else " ".join(items)

    pred_series = df_test.groupby(order_key).apply(lambda g: to_pred_string(g, best_thr))
    sub_df = pred_series.reset_index()
    sub_df.columns = [order_key, "products"]  # (order_id, products)

    # 저장
    detailed_pred_path = os.path.join(OUT_DIR, "test_predictions_detailed.csv")
    cols_det = [c for c in [order_col, member_col, product_col] if c] + ["proba_A","proba_B","proba_ens"]
    if has_label_test: cols_det.append(target_col)
    df_test[cols_det].to_csv(detailed_pred_path, index=False)

    order_pred_path = os.path.join(OUT_DIR, f"test_predictions_orders_threshold_{best_thr:.2f}.csv")
    sub_df.to_csv(order_pred_path, index=False)

    # 메타 파일 업데이트
    with open(meta_path, "w") as f:
        json.dump({
            'tree_method': 'hist',
            'device': 'cuda' if cfg_A.tree_method=='gpu_hist' else 'cpu',
            'cfg_A': asdict(cfg_A),
            'cfg_B': asdict(cfg_B),
            'features_A': res_A['features'],
            'features_B': res_B['features'],
            'report_oof': report,
            'best_threshold': best_thr,
            'columns': {
                'target_col': target_col,
                'order_col': order_col,
                'member_col': member_col,
                'product_col': product_col,
                'role_col': role_col,
                'role_train_col': role_train_col,
                'role_test_col': role_test_col
            },
            'artifacts': {
                'oof_predictions': os.path.basename(os.path.join(OUT_DIR, "oof_predictions_instacart.csv")),
                'test_predictions_detailed': os.path.basename(detailed_pred_path),
                'test_predictions_orders': os.path.basename(order_pred_path),
                'fold_models_A': [f"xgb_A_fold{i}.json" for i in range(1, NFOLDS+1)],
                'fold_models_B': [f"xgb_B_fold{i}.json" for i in range(1, NFOLDS+1)]
            }
        }, f, indent=2, ensure_ascii=False)

    print("저장 완료:")
    print(" - 상세 예측(행 단위):", detailed_pred_path)
    print(" - 주문별 예측(제출 포맷):", order_pred_path)

[TEST] Ensemble | Thr=0.32 | F1=0.70479 | AUC=0.72942 | Logloss=0.59356


/tmp/ipython-input-1073343652.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pred_series = df_test.groupby(order_key).apply(lambda g: to_pred_string(g, best_thr))


저장 완료:
 - 상세 예측(행 단위): /content/drive/MyDrive/data/instacart/test_predictions_detailed.csv
 - 주문별 예측(제출 포맷): /content/drive/MyDrive/data/instacart/test_predictions_orders_threshold_0.32.csv


In [15]:
# ===== 8. 피처 중요도 저장 — Booster 호환 =====
def save_importance_booster(models: List[xgb.Booster], features: List[str], out_path: str):
    # Booster가 f0,f1,... 로 저장할 수 있어 feature_names를 강제로 세팅
    # (xgb.train 시 DMatrix에 feature_names를 넣었으므로 보통 이름이 유지됩니다.)
    agg = np.zeros(len(features), dtype=np.float64)
    for bst in models:
        fmap = bst.get_score(importance_type='gain')
        # fmap 키가 'f0' 형태거나 실제 피처명일 수 있음 → 둘 다 처리
        for i, f in enumerate(features):
            agg[i] += float(fmap.get(f, 0.0) or 0.0)
            # f가 없고 f{i} 키만 있는 경우 대응
            if agg[i] == 0.0:
                alt = f"f{i}"
                agg[i] += float(fmap.get(alt, 0.0) or 0.0)
    agg /= max(1, len(models))
    imp = pd.DataFrame({'feature': features, 'gain': agg}).sort_values('gain', ascending=False)
    imp.to_csv(out_path, index=False)
    print("저장:", out_path)

save_importance_booster(res_A['models'], res_A['features'], os.path.join(OUT_DIR, "feature_importance_A.csv"))
save_importance_booster(res_B['models'], res_B['features'], os.path.join(OUT_DIR, "feature_importance_B.csv"))


저장: /content/drive/MyDrive/data/instacart/feature_importance_A.csv
저장: /content/drive/MyDrive/data/instacart/feature_importance_B.csv
